# Web Scraping con Selenium



##### Extrae de las películas en cartelera datos. De ellas vamos a extraer la siguiente información:
- ##### Fecha de estreno
- ##### URL
- ##### Datos principales, como hemos hecho al principio
- ##### Nota media
- ##### Cantidad de votos
- ##### Críticas profesionales buenas, regulares y malas
- ##### Cinco primeras críticas

Importación de las librerías

In [64]:
# Para la manipulación de datos
import pandas as pd

# Servicio y driver de Chrome de Selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Las opciones que vamos a tener para buscar elementos
from selenium.webdriver.common.by import By

# Para cuando queramos mandar pulsaciones de teclado
from selenium.webdriver.common.keys import Keys

# Hacemos que espere
import time

# Importaciones para esperas explícitas (mejor práctica que time.sleep)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Importamos undetected-chromedriver para evitar el captcha
import undetected_chromedriver as uc

# Importamos excepciones comunes de Selenium para manejo de errores
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementClickInterceptedException

In [65]:
service = Service(ChromeDriverManager().install())

In [66]:
from selenium import webdriver

driver = webdriver.Chrome()
print(driver.capabilities["browserVersion"])
driver.quit()

146.0.7680.154


Creamos el driver para controlar el navegador

In [67]:
import undetected_chromedriver as uc

driver = uc.Chrome(
    browser_executable_path=r"C:\Program Files\Google\Chrome\Application\chrome.exe",
    service=service,
    use_subprocess=False,
    headless=False,
)

RECUERDA:

##### Beginner Selenium Cheatsheet:
Sacar un elemento:
- element = driver.find_element(by, value)

Sacar varios elementos:
- element = driver.find_elements(by, value)

Sacar atributos:
- attribute = element.--el atributo--
- attribute = element.get_attribute(--el atributo--)

Hacer click:
- element.click()

Teclear:
- element.send_keys()

Gestión de pestañas:
- driver.switch_to.window(driver.window_handles[-1])
- driver.get(url)
- driver.close()

Accedemos a la página principal

In [68]:
url = "https://www.filmaffinity.com"
driver.get(url)

Aceptamos el pop-up de ser necesario

In [69]:
# tu código
element_by_tag = driver.find_elements(By.TAG_NAME, "button")
print(element_by_tag)

time.sleep(1)
accept = driver.find_element(By.ID, "accept-btn").click()
time.sleep(1)

cartelera = driver.find_element(By.LINK_TEXT, "Películas en cartelera")
cartelera.click()

[<undetected_chromedriver.webelement.WebElement (session="b5db6d51ea42cfffcf8c955246dcf636", element="f.C98EBAB8F84E495FBBDE745AF0966838.d.1693C977A2B76248F73372BBA0D6AB60.e.145")>, <undetected_chromedriver.webelement.WebElement (session="b5db6d51ea42cfffcf8c955246dcf636", element="f.C98EBAB8F84E495FBBDE745AF0966838.d.1693C977A2B76248F73372BBA0D6AB60.e.146")>]


Hacemos una función que devuelva en un diccionario todos los datos de las películas, salvo la fecha de estreno y la url

Parámetros: url y fecha de estreno
Salida: Diccionario con todos los datos

In [70]:
# wait = WebDriverWait(driver, 5)
# wait.until(EC.presence_of_element_located((By.CLASS_NAME, "movie-title")))
# movie_links = driver.find_elements(By.CLASS_NAME, "movie-title")

# print(f"Elementos encontrados: {len(movie_links)}")

# for i, elem in enumerate(movie_links):
#     print(f"[{i}] Tag: {elem.tag_name} | Texto: {elem.text} | href: {elem.get_attribute('href')}")

In [71]:
def get_datos_pelicula(driver, url, estreno):
    all_movies = {}

    all_movies["fecha_estreno"] = estreno
    all_movies["url"] = url

    driver.get(url)
    time.sleep(2)

    # Datos principales
    data = driver.find_element(By.CLASS_NAME, "movie-info")
    dts = data.find_elements(By.TAG_NAME, "dt")
    dds = data.find_elements(By.TAG_NAME, "dd")

    # Añado los dts y dds en el diccionario
    for x, y in zip(dts, dds):
        all_movies[x.text.strip()] = y.text.strip()

    nota_media = driver.find_elements(By.ID, value="movie-rat-avg")
    if len(nota_media) > 0:
        all_movies["nota_media"] = nota_media[0].text

    cantidad_votos = driver.find_elements(By.CSS_SELECTOR, value="#movie-count-rat span")
    if len(cantidad_votos) > 0:
        all_movies["cantidad_votos"] = cantidad_votos[0].text

    n_criticas = driver.find_elements(By.CSS_SELECTOR, value="#right-column > a > div > div.body > div > div.legend-wrapper .leg")

    if len(n_criticas) > 0:
        positivas = n_criticas[0].text
        all_movies["positivas"] = positivas

        regulares = n_criticas[1].text
        all_movies["regulares"] = regulares

        negativas = n_criticas[2].text
        all_movies["negativas"] = negativas

    criticas = driver.find_elements(By.CSS_SELECTOR, value="ul#pro-reviews li")
    i = 0

    while i < 5 and i < len(criticas):
        critica = criticas[i].find_element(by=By.CSS_SELECTOR, value='div div').text
        all_movies['critica_'+str(i)] = critica

        i += 1

    return all_movies
    

In [72]:
# prueba = get_datos_pelicula(driver, "https://www.filmaffinity.com/es/film618375.html", "fecha")
# prueba
# all_movies = []

# Sacamos las fechas de estreno dentro de cartelera
# fecha_estreno = driver.find_elements(By.CLASS_NAME, "release-text")

# Sacamos los links de acceso a todas las peliculas de cartelera
movie_link = driver.find_elements(by=By.CSS_SELECTOR, value='div.col a.poster-wrap')

# Creamos una lista vacía a la que le añadimos los links
hrefs = []

for elemento in movie_link:

    estreno = elemento.find_element(by=By.CSS_SELECTOR, value='small.release-text').text
    link_pelicula = {
            'título': elemento.get_attribute('title'),
            'url': elemento.get_attribute('href'),
            'estreno': estreno.replace("\n", " ")
    }

    hrefs.append(link_pelicula)

print(f"Links guardados: {len(hrefs)}")

# iteramos la lista de links para acceder a cada una y aplicar la funcion definida previamente
df = None
for href in hrefs:
    datos_pelicula = get_datos_pelicula(driver, href["url"], href["estreno"])
    print(f"Navegando a: {href}")

    if df is None:
        df = pd.DataFrame(columns=datos_pelicula.keys())
    
    df = pd.concat([df, pd.DataFrame(datos_pelicula, index = [0])], ignore_index=True)
    print(f"Añadida {datos_pelicula['Título original']}")

#     data = driver.find_element(By.CLASS_NAME, "movie-info")
#     dts = data.find_elements(By.TAG_NAME, "dt")
#     dds = data.find_elements(By.TAG_NAME, "dd")

#     pelicula = {}
#     for x, y in zip(dts, dds):
#         pelicula[x.text.strip()] = y.text.strip()

#     all_movies.append(pelicula)

#     driver.back()
#     wait.until(EC.presence_of_element_located((By.CLASS_NAME, "movie-title")))

# print(all_movies)

Links guardados: 60
Navegando a: {'título': 'Amarga Navidad ', 'url': 'https://www.filmaffinity.com/es/film310439.html', 'estreno': '20 mar.'}
Añadida Amarga Navidad
Navegando a: {'título': 'Whistle: El silbido del mal ', 'url': 'https://www.filmaffinity.com/es/film261988.html', 'estreno': '20 mar.'}
Añadida Whistle
Navegando a: {'título': 'Una hija en Tokio ', 'url': 'https://www.filmaffinity.com/es/film691891.html', 'estreno': '20 mar.'}
Añadida Une part manquanteaka
Navegando a: {'título': 'Elegir mi vida ', 'url': 'https://www.filmaffinity.com/es/film913052.html', 'estreno': '20 mar.'}
Añadida Partir un jouraka
Navegando a: {'título': 'Tafiti y sus amigos ', 'url': 'https://www.filmaffinity.com/es/film448750.html', 'estreno': '20 mar.'}
Añadida Tafiti - Ab durch die Wüste
Navegando a: {'título': 'Your Name. ', 'url': 'https://www.filmaffinity.com/es/film307521.html', 'estreno': '20 mar.'}
Añadida Kimi no Na wa. (Your Name.)
Navegando a: {'título': 'La sonrisa del mal ', 'url': 'htt

In [73]:
df

,fecha_estreno,url,Título original,Año,Duración,País,Dirección,Guion,Reparto,Música,...,Sinopsis,nota_media,cantidad_votos,critica_0,critica_1,critica_2,critica_3,critica_4,,Grupos
0,20 mar.,https://www.filmaffinity.com/es/film310439.html,Amarga Navidad,2026,111 min.,España,Pedro Almodóvar,Pedro Almodóvar,Bárbara Lennie\nLeonardo Sbaraglia\nAitana Sán...,Alberto Iglesias. Canción: Amaia,...,Elsa es una directora de publicidad cuya madre...,"6,2",1.685,"""Feliz choque inesperado: Almodóvar lanza su v...","""Almodóvar y las razones, que son emociones, d...","""Almodóvar se encomienda a la autoficción en u...","""Un melodrama preciso y sereno (...) fluye a t...","""Almodóvar pone en imágenes un guion modélico,...",NaN,NaN
1,20 mar.,https://www.filmaffinity.com/es/film261988.html,Whistle,2025,100 min.,Canadá,Corin Hardy,Owen Egerton,Dafne Keen\nSophie Nélisse\nNick Frost\nPercy ...,Doomphonic,...,Un grupo de estudiantes inadaptados encuentra ...,"4,7",370,"""La originalidad no es lo que busca “Whistle”,...","""No sorprende tanto por lo que cuenta como por...","""La trivialidad del desarrollo narrativo, no o...","""Visibiliza la solvencia del director (...) en...","""Whistle cumple las expectativas ofreciendo un...",NaN,NaN
2,20 mar.,https://www.filmaffinity.com/es/film691891.html,Une part manquanteaka,2024,98 min.,Francia,Guillaume Senez,"Guillaume Senez, Jean Denizot",Romain Duris\nJudith Chemla\nMei Cirne-Masuki\...,Olivier Marguerit,...,"Todos los días, Jay conduce su taxi por Tokio ...","6,9",232,"""La película da pocas explicaciones en general...","""Con tacto, la pareja Senez-Duris acierta a co...","""Logra maximizar el impacto emocional de sus m...","""Una cinta delicada y bien escrita (...) Lo me...","""Un filme delicado y conmovedor. De sobriedad ...",,NaN
3,20 mar.,https://www.filmaffinity.com/es/film913052.html,Partir un jouraka,2025,94 min.,Francia,Amélie Bonnin,"Amélie Bonnin, Dimitri Lucas",Juliette Armanet\nBastien Bouillon\nFrançois R...,"Keren Ann, Gonzales, Germain Izydorczyk, Theo ...",...,Cécile está a punto de cumplir su sueño de abr...,"5,9",83,"""Bonnin convierte la nostalgia en el argumento...","""Como película, es de esas que no molestan, qu...","""Una oda a la cultura pop francesa pasada por ...","""Es agradable de ver por el carisma de los act...","""Ni los toques de originalidad del filme nos d...",,NaN
4,20 mar.,https://www.filmaffinity.com/es/film448750.html,Tafiti - Ab durch die Wüste,2025,81 min.,Alemania,"Nina Wels, Timo Berg","Julia Boehme, Nicholas Hause",Cosima Henman\nVoz\nSteve Hudson\nVoz\nDela Da...,Carsten Rocker,...,"Cuando Tafiti, una joven suricata, conoce a Br...",NaN,NaN,"""Son adorables, pero este dúo de suricato y ce...",NaN,NaN,NaN,NaN,NaN,NaN
5,20 mar.,https://www.filmaffinity.com/es/film307521.html,Kimi no Na wa. (Your Name.),2016,106 min.,Japón,Makoto Shinkai,Makoto Shinkai,Animación,Radwimps,...,Taki y Mitsuha descubren un día que durante el...,"7,8",29.580,"""Your Name"" fue un éxito rotundo en taquilla, ...","""Tanto el luminoso estilo del animador como su...","""La película, profundamente adulta y absolutam...","""Fastuoso relato entre realista y fantástico (...","""Your Name es la obra maestra de Makoto Shinka...",NaN,NaN
6,20 mar.,https://www.filmaffinity.com/es/film359278.html,La valle dei sorrisi,2025,122 min.,Italia,Paolo Strippoli,"Jacopo Del Giudice, Paolo Strippoli, Milo Tissone",Michele Riondino\nGiulio Feltri\nPaolo Pierobo...,"Federico Bisozzi, Davide Tomat",...,Remis es un pequeño pueblo enclavado en un val...,"6,5",169,"""Una escalofriante cinta sobre el fanatismo, e...","""Es deliberadamente confusa en buena parte de ...","""Es un híbrido de terror intelectual, sin sust...","""Una sorprendente resurrección del cine de ter...","""Una pesadilla implacable y gráfica (...) 'La ...",NaN,NaN
7,20 mar.,https://www.filmaffinity.com/es/film718849.html,"Nos veremos esta noche, mi amor",2026,79 min.,México,Francisco Arasanz,Francisco Arasanz,"Sergio María, María Monroy, Pa

Probamos la función que hemos hecho. Aquí tienes un enlace de prueba: https://www.filmaffinity.com/es/film599984.html

In [8]:
prueba = get_datos_pelicula(driver, "https://www.filmaffinity.com/es/film618375.html", "fecha")
prueba

{'fecha_estreno': 'fecha',
 'url': 'https://www.filmaffinity.com/es/film618375.html',
 'Título original': 'Oblivion',
 'Año': '2013',
 'Duración': '126 min.',
 'País': ' Estados Unidos',
 'Dirección': 'Joseph Kosinski',
 'Guion': 'Joseph Kosinski, Michael Arndt, Karl Gajdusek. Cómic: Joseph Kosinski, Arvid Nelson',
 'Reparto': 'Tom Cruise\nAndrea Riseborough\nOlga Kurylenko\nMorgan Freeman\nNikolaj Coster-Waldau\nZoe Bell',
 'Música': 'Anthony Gonzalez, M83, Joseph Trapanese',
 'Fotografía': 'Claudio Miranda',
 'Compañías': 'Universal Pictures, Chernin Entertainment, Relativity Studios, Monolith Pictures, Radical Studios',
 'Género': 'Ciencia ficción. Intriga | Futuro postapocalíptico. Distopía. Cómic',
 'Sinopsis': 'Año 2073. Hace más de 60 años la Tierra fue atacada; se ganó la guerra, pero la mitad del planeta quedó destruido, y todos los seres humanos fueron evacuados. Jack Harper (Tom Cruise), un antiguo marine, es uno de los últimos hombres que la habitan. Es un ingeniero de Dron

Entramos en el link de las películas en cartelera

In [ ]:
# tu código

In [ ]:
# tu código

[<undetected_chromedriver.webelement.WebElement (session="88d13dc2b36b434bc6ba970c4da8d75a", element="f.1B45C29A1E40EB3D37BD0DBD2F1F28D6.d.3297110C6BC4EC58CCD60B4D9B7DB636.e.7913")>,
 <undetected_chromedriver.webelement.WebElement (session="88d13dc2b36b434bc6ba970c4da8d75a", element="f.1B45C29A1E40EB3D37BD0DBD2F1F28D6.d.3297110C6BC4EC58CCD60B4D9B7DB636.e.9469")>,
 <undetected_chromedriver.webelement.WebElement (session="88d13dc2b36b434bc6ba970c4da8d75a", element="f.1B45C29A1E40EB3D37BD0DBD2F1F28D6.d.3297110C6BC4EC58CCD60B4D9B7DB636.e.9492")>,
 <undetected_chromedriver.webelement.WebElement (session="88d13dc2b36b434bc6ba970c4da8d75a", element="f.1B45C29A1E40EB3D37BD0DBD2F1F28D6.d.3297110C6BC4EC58CCD60B4D9B7DB636.e.9507")>,
 <undetected_chromedriver.webelement.WebElement (session="88d13dc2b36b434bc6ba970c4da8d75a", element="f.1B45C29A1E40EB3D37BD0DBD2F1F28D6.d.3297110C6BC4EC58CCD60B4D9B7DB636.e.9528")>]

Sacamos todas las películas y llamamos a la función con cada película

In [ ]:
# tu código


In [43]:
links

[{'título': 'Dhurandhar: The Revenge ',
  'url': 'https://www.filmaffinity.com/es/film343952.html',
  'estreno': '19 mar.'},
 {'título': 'Torrente presidente ',
  'url': 'https://www.filmaffinity.com/es/film453739.html',
  'estreno': '13 mar.'},
 {'título': 'Águilas de El Cairo ',
  'url': 'https://www.filmaffinity.com/es/film314137.html',
  'estreno': '13 mar.'},
 {'título': 'El testamento de Ann Lee ',
  'url': 'https://www.filmaffinity.com/es/film842010.html',
  'estreno': '13 mar.'},
 {'título': 'El arquitecto ',
  'url': 'https://www.filmaffinity.com/es/film855002.html',
  'estreno': '13 mar.'},
 {'título': 'No te queda otra ',
  'url': 'https://www.filmaffinity.com/es/film732131.html',
  'estreno': '13 mar.'},
 {'título': 'La hija pequeña ',
  'url': 'https://www.filmaffinity.com/es/film781408.html',
  'estreno': '13 mar.'},
 {'título': 'Las locas del obelisco ',
  'url': 'https://www.filmaffinity.com/es/film310301.html',
  'estreno': '13 mar.'},
 {'título': '¡La novia! ',
  'url

Ahora usamos los links para llamar a la funcion y sacar los datos

In [ ]:
# tu código

Añadida Dhurandhar: The Revengeaka
Añadida Torrente presidenteaka
Añadida Eagles of the Republicaka
Añadida The Testament of Ann Lee
Añadida L'inconnu de la Grande Archeaka
Añadida Aimons-nous vivants
Añadida La petite dernière
Añadida Las locas del obelisco
Añadida The Bride!
Añadida Hoppers
Añadida Le Mage du Kremlinaka
Añadida Den Sidste Vikingaka
Añadida Cleaner
Añadida Elevation
Añadida Caminando con el diablo
Añadida Pillion
Añadida Aves de corral
Añadida Marcel et Monsieur Pagnolaka
Añadida América Hispana. El legado de las olas
Añadida The Face of the Faceless
Añadida My Father's Shadow
Añadida Karavanaka
Añadida Scream 7
Añadida Scarletaka
Añadida Jean Valjeanaka
Añadida Sorry, Baby
Añadida EPiC: Elvis Presley in Concert
Añadida Dalloway
Añadida Iron Lung
Añadida Orwell: 2+2=5
Añadida Islas
Añadida Bergersaka
Añadida Manas
Añadida Greenland 2: Migration
Añadida O Agente Secretoaka
Añadida El fantasma de mi mujer
Añadida Shelby Oaks
Añadida Amélie et la Métaphysique des tubesak

In [45]:
df

,fecha_estreno,url,Título original,,Año,Duración,País,Dirección,Guion,Reparto,...,Género,Sinopsis,Grupos,nota_media,cantidad_votos,critica_0,critica_1,critica_2,critica_3,critica_4
0,19 mar.,https://www.filmaffinity.com/es/film343952.html,Dhurandhar: The Revengeaka,,2026,210 min.,India,Aditya Dhar,Aditya Dhar,Ranveer Singh\nSanjay Dutt\nAkshaye Khanna\nMa...,...,Acción. Thriller | Secuela. Espionaje,"Bajo la identidad de Hamza Ali Mazari, el espí...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,13 mar.,https://www.filmaffinity.com/es/film453739.html,Torrente presidenteaka,,2026,102 min.,España,Santiago Segura,Santiago Segura,Santiago Segura\nGabino Diego\nRamón Langa\nCa...,...,Comedia | Comedia negra. Sátira. Política. Sec...,"José Luis Torrente, siempre convencido de ser ...",Torrente,"5,9",3.144,"""Una sátira tan ocurrente y divertida como opo...","""Soez, excesiva y por momentos muy divertida. ...","""Cuñadismo para todos los públicos (...) No ha...","""Una comedia facilona (...) sin tanto colmillo...","""Es fiel a la saga, pero va un poco más allá (..."
2,13 mar.,https://www.filmaffinity.com/es/film314137.html,Eagles of the Republicaka,,2025,127 min.,Suecia,Tarik Saleh,Tarik Saleh,Fares Fares\nLyna Khoudri\nZineb Triki\nAmr Wa...,...,Thriller. Drama | Cine dentro del cine. Política,George Fahmy (Fares Fares) el actor más querid...,Trilogía de El Cairo,"6,4",169,"""Cierra la trilogía de Tarik Saleh sobre la ci...","""En su núcleo central deambula un tanto entre ...","""De sus imágenes emerge un duro retrato crític...","""El argumento que trata es interesante, tiene ...","""Se ve lastrada por la tosquedad como narrador..."
3,13 mar.,https://www.filmaffinity.com/es/film842010.html,The Testament of Ann Lee,NaN,2025,136 min.,Reino Unido,Mona Fastvold,"Mona Fastvold, Brady Corbet",Amanda Seyfried\nThomasin McKenzie\nLewis Pull...,...,Drama. Musical | Basado en hechos reales. Sigl...,"Inspirada en hechos reales, se centra en la lí...",NaN,"5,9",345,"""Un filme impresionante de principio a fin (.....","""El fervor religioso (...) incluso la épica de...","""La directora lo quiere todo (bien) y todo lo ...","""Amanda Seyfried hace un trabajo excelente en ...","""Resulta difícil no sentir una fascinación int..."
4,13 mar.,https://www.filmaffinity.com/es/film855002.html,L'inconnu de la Grande Archeaka,,2025,105 min.,Francia,Stéphane Demoustier,Stéphane Demoustier. Novela: Laurence Cossé,Claes Bang\nSidse Babett Knudsen\nXavier Dolan...,...,Drama | Basado en hechos reales. Biográfico. A...,Año 1983. El nuevo gobierno francés de Françoi...,NaN,"6,4",130,"""Una película arquitectónicamente brillante. (...","""El nivel de detalle de Demoustier, lastra el ...","""Lo peor es pensar que es un biopic convencion...","""Solo para expertos y profesionales de la arqu...","""Suspense, humor y un brillante reparto conviv..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,24 oct.,https://www.filmaffinity.com/es/film584300.html,Los domingos,NaN,2025,110 min.,España,Alauda Ruiz de Azúa,Alauda Ruiz de Azúa,Blanca Soroa\nPatricia López Arnaiz\nMiguel Ga...,...,Drama | Religión. Familia. Monjas. Adolescencia,"Ainara (Blanca Soroa), una joven idealista y b...",NaN,"7,3",17.841,"""La nueva propuesta de la directora (...), fie...","""Ruiz de Azúa infunde fe en el gran cine (...)...","""Una película tan a contrapelo y tan luminosa,...","""Hay que celebrar el rigor cartesiano, equilib...","""Confirma que Ruiz de Azúa es una directora de..."
65,6 jun.,https://www.filmaffinity.com/es/film780154.html,Sirât,NaN,2025,114 min.,España,Oliver Laxe,"Oliver Laxe, Santiago Fillol",Sergi López\nBruno Núñez\nJade Oukid\nStefania...,...,Drama. Intriga. Thriller | Road Movie. Secuest...,Un hombre (Sergi López) y su hijo (Bruno Núñez...,NaN,"6,2",25.289,"""Laxe revienta Cannes con el trágico y brutal ...","""Una propuesta tan radical y honda como sorpre...","""Brillante ejercicio de hipnosis (...) una obr...","""Durante todo el metraje estoy